## BMLL Notebook Educational Series
### Options Data Processing: OPRA Snapshots and Resampling

In [1]:
import os
from functools import lru_cache
import sys

def install_snowflake_connector():
    os.system(sys.executable + " -m pip install snowflake-connector-python[pandas] snowflake-sqlalchemy")

@lru_cache(5)
def snowflake_connection(
    database="PROD_DERIVED_DATA_DB",
    schema="PUBLIC",
    warehouse="PROD_DERIVED_DEFAULT_WH",
    role="PROD_SALES_RO_ROLE",
    user=None,
    password=None,
    use_secret_storage=True,
    sso=None,
):
    import snowflake
    import snowflake.connector
    try:
        from snowflake.snowpark import Session as SnowparkSession
    except Exception:
        SnowparkSession = None
    try:
        import bmll2
        stor = bmll2.storage_paths()["user"]
        os.environ["SNOWFLAKE_USER"] = stor["prefix"].split("/")[1]
        os.environ["SNOWFLAKE_USE_SSO"] = "1"
    except Exception:
        pass
    if sso is None:
        sso = os.environ.get("SNOWFLAKE_USE_SSO")

    password = os.environ.get("SNOWFLAKE_PASSWORD", password)
    if not password:
        try:
            import secretstorage
            connection = secretstorage.dbus_init()
            collection = secretstorage.get_default_collection(connection)
            for item in collection.search_items({"name": "default_snowflake", "system": "bmll"}):
                password = item.get_secret()
        except Exception as e:
            print(f"Error connecting to Secret Service: {e}")
    else:
        try:
            import secretstorage
            attributes = {"name": "default_snowflake", "system": "bmll"}
            label = "default_snowflake"
            secret = password
            connection = secretstorage.dbus_init()
            collection = secretstorage.get_default_collection(connection)
            if collection.is_locked():
                unlocked = collection.unlock()
                if unlocked:
                    print("Collection unlocked successfully.")
                else:
                    print("Failed to unlock the collection.")
            else:
                print("Collection is already unlocked.")
            collection.create_item(label, attributes, secret, replace=True)
        except Exception as e:
            print(e)

    if not sso:
        connargs = dict(
            user=os.environ.get("SNOWFLAKE_USER", user),
            password=password,
            account=os.environ.get("SNOWFLAKE_ACCOUNT", "tib96089"),
            region=os.environ.get("SNOWFLAKE_REGION", "us-east-1"),
            role=role,
            database=database,
            schema=schema,
            warehouse=warehouse,
        )
    else:
        connargs = dict(
            user=os.environ.get("SNOWFLAKE_USER", user),
            account=os.environ.get("SNOWFLAKE_ACCOUNT", "tib96089"),
            region=os.environ.get("SNOWFLAKE_REGION", "us-east-1"),
            role=role,
            database=database,
            schema=schema,
            warehouse=warehouse,
            authenticator="externalbrowser",
        )
    if SnowparkSession is not None:
        return SnowparkSession.builder.configs(connargs).create()
    return snowflake.connector.connect(**connargs)

session = snowflake_connection()
if not hasattr(session, "sql"):
    class _SFResult:
        def __init__(self, cursor):
            self._cursor = cursor

        def to_pandas(self):
            try:
                return self._cursor.fetch_pandas_all()
            except Exception:
                import pandas as pd
                rows = self._cursor.fetchall()
                cols = [c[0] for c in self._cursor.description] if self._cursor.description else []
                return pd.DataFrame(rows, columns=cols)

    class _SFSessionShim:
        def __init__(self, conn):
            self._conn = conn

        def sql(self, query):
            cur = self._conn.cursor()
            cur.execute(query)
            return _SFResult(cur)

    session = _SFSessionShim(session)

/home/bertrand/miniconda3/lib/python3.11/site-packages/bmll2/_internals/api/configure.py:116: UserWarning: BMLL configuration file (/tmp/.nb_extension_instance_vars.json) not found. BMLL data and S3 storage will not be reachable.
  warnings.warn(f"{msg} BMLL data and S3 storage will not be reachable.", UserWarning)


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://accounts.google.com/o/saml2/idp?idpid=C0433grw8&SAMLRequest=lZJPc5swEMW%2FCqOeQQITx9YYe1w7aek4rWvjHHJThIzVCIlqRUj66Sv8ZyY9JDM9wDDiPf129%2B1k9lKr4FlYkEZnKI4ICoTmppS6ytCuuA1HKADHdMmU0SJDrwLQbDoBVquGzlt30BvxuxXgAn%2BRBtr%2FyFBrNTUMJFDNagHUcbqd361oEhHKAIR1HofOlhKkZx2cayjGXddF3SAytsIJIQSTMfaqXvIJvUE0HzMaa5zhRl0sL76ndxAxJmmP8ApPWJ%2BNn6U%2BjeAjyuNJBPRrUazD9Y9tgYL5pbuF0dDWwm6FfZZc7DarUwHgK3islQr7lxP8EIE23V6xJ8FN3bTO3xj5L7wXJVamkn5O%2BTJDzZMsrx9EW5BN%2BufXtatvrkS%2Bs9%2FG%2BZCPm%2BJerrb7Zvkl%2FWl2CdzkHAX3l1STPtUcoBW57rN0%2Fogkw5CMQnJdxFc0HlIyjNJR%2BoCCpc9SauaOzkvBjHPTagdRZUylxLE%2Bg%2FsgEizLZuYfWWYLkg4Gle1G6LQg9Mi00%2F9pe4LfOs979t2PPl%2BujZL8Nbg1tmbu%2FWTiKD6eyDLcH6VU1EyqeVlaAeATUsp0CyuY8%2BvsbCsQnp6o%2Fy709C8%3D&RelayState=ver%3A3-hint%3A10

In [2]:
underlying_ticker = 'SPY'
date = '2025-02-07'

### OPRA: The Options Data Foundation

**OPRA (Options Price Reporting Authority)** aggregates real-time quote and trade data from all US options exchanges:
- **8+ options exchanges** report simultaneously (CBOE, NASDAQ, NYSE Arca, etc.)
- **Millions of quotes per day** across all option contracts (calls, puts, multiple expirations, strikes)
- **High frequency updates** every time bid/ask or last trade changes

**Why Resampling Matters**:
- Raw OPRA data is tick-by-tick (every change = one record)
- 1 million+ records per underlying per day is computationally intensive
- Traders need snapshots at regular intervals (1-minute, 5-minute charts)
- Resampling reduces data volume while preserving key information

**Use Cases**:
- **Greeks calculation**: Compute option Greeks (delta, gamma, vega) at regular intervals
- **Volatility surface analysis**: Monitor how implied volatility changes across strikes/expirations
- **Option flow analysis**: Aggregate unusual activity (large trades, unusual order sizes)
- **Portfolio risk monitoring**: Periodically recalculate portfolio Greeks and exposures
- **Backtesting**: Resample to regular time intervals for strategy simulations

This notebook demonstrates efficient processing of large OPRA datasets using Polars
(for fast in-memory operations) and resampling to create actionable option snapshots.

In [3]:
import pandas as pd

In [4]:
# Expect an existing Snowpark session named `session` in Snowflake worksheets.

In [5]:
# Define the study set: a list of stock tickers to focus on
ticker_set = ['SPY', 'QQQ', 'TSLA', 'AAPL', 'IWM', 'NVDA', 'AMD', 'AMZN']

# Set the date for which you want to fetch data
date = "2025-02-07"

In [6]:
# Define the columns we want to extract
cols = ["Ticker", "UnderlyingTicker", "LocalTimestamp", "BestBidPrice", "BestAskPrice"]

# Fetch market data from OPRA using bmll2's get_market_data function.
# Parameters: source is OPRA, date is the target date, data type is 'nbbo', and lazy loading is enabled.

In [7]:
# Convert the date to Unix timestamp and adjust for the target time window
ts = int((pd.Timestamp(date).timestamp() + (3600 * 12)) * 10**9)  # 12 hours offset in nanoseconds
te = ts + 60 * 10**9  # Add 10 minutes (in nanoseconds) to the start time

# Filter the dataframe:
# - Filter for LocalTimestamp between 'ts' and 'te'
# - Filter for rows where UnderlyingTicker is in our 'ticker_set'
# - Select the relevant columns

In [8]:
opra_sql = f"""
    SELECT
        TICKER AS "Ticker",
        UNDERLYING_TICKER AS "UnderlyingTicker",
        LOCAL_TIMESTAMP AS "LocalTimestamp",
        BEST_BID_PRICE AS "BestBidPrice",
        BEST_ASK_PRICE AS "BestAskPrice"
    FROM "PROD_L2_CBBO_DB"."PUBLIC"."OPRA_TRADES"
    WHERE DATE = '{date}'
      AND UNDERLYING_TICKER IN ({", ".join(f"'{t}'" for t in ticker_set)})
      AND LOCAL_TIMESTAMP > {ts}
      AND LOCAL_TIMESTAMP < {te}
"""

df3 = session.sql(opra_sql).to_pandas()
df3 = df3.sort_values("LocalTimestamp")
result_df = df3.groupby(["Ticker"], as_index=False).first()
print(result_df)

                     Ticker UnderlyingTicker       LocalTimestamp  \
0     AAPL  250207C00227500             AAPL  1738929609914511360   
1     AAPL  250207C00230000             AAPL  1738929600367909632   
2     AAPL  250207C00232500             AAPL  1738929605392888064   
3     AAPL  250207C00235000             AAPL  1738929631326515968   
4     AAPL  250207P00217500             AAPL  1738929629438145536   
...                     ...              ...                  ...   
2041  TSLA  270617C00700000             TSLA  1738929603048004352   
2042  TSLA  270617P00140000             TSLA  1738929602998008576   
2043  TSLA  270617P00180000             TSLA  1738929602999211776   
2044  TSLA  270617P00300000             TSLA  1738929601723908864   
2045  TSLA  270617P00380000             TSLA  1738929650832843520   

      BestBidPrice  BestAskPrice  
0             1.66          1.74  
1             0.35          0.36  
2             0.06          0.07  
3             0.01          0.0